## Ripple data analysis

Contains Ripple data collected from Gian over two sessions.


In [1]:
# Loading/Import related packages
import sys
import os

#Usual suspects
import pandas as pd
import numpy as np
import json
import pickle as pkl
import matplotlib.pyplot as plt

#Extras for plotting
from matplotlib.patches import Patch

#Extra for typing
from collections import defaultdict

# Needed to point the importer to the src folder
sys.path.insert(0,'/Users/lizkal/Library/CloudStorage/SynologyDrive-Personal/MotorUnitSuite')

#Decomposition/Processing Imports
from src.muniverse.algorithms.cbss import CBSS

In [2]:
REPO_DIR  = os.path.abspath(os.getcwd())
INPUT_DIR = os.path.join(REPO_DIR, 'data')
OUTPUT_DIR = os.path.join(REPO_DIR, 'results')

combined_data_1 = 'emg_recording_giuan_tewst_20260826_122524.pkl'



In [3]:
def load_simulation_data(input_dir, filename):

    """
    Load the simulation data from a pickle file.

    Parameters:
    - input_dir: str, the directory where the pickle file is located.
    - filename: str, the name of the pickle file.

    Returns:
    - emg_array: dict, the loaded simulation EMG data. 
    Normally contains 320 channels, here downsampled to the first 8x8 array to make decomposition faster .
    """
    file_path = os.path.join(input_dir, filename)

    # Check if the file exists before attempting to load it
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist. Please ensure the input directory and filename are correct.")

    result_pandas = pd.read_pickle(file_path)
    print("Loaded data from:", file_path)
    print(result_pandas.keys())
    print(result_pandas['data'].shape)

    emg_array = result_pandas['data']
    #These two arrays contain the intended poses and durations for each pose
    # We will use this information later in the analysis, see below in the Decomposition Section
    poses = result_pandas['metadata']['poses_intended']
    durations = result_pandas['metadata']['durations']


    #NOTE: a tuple, not a set: a set has no order, so the unpacking below would be random
    return emg_array, poses, durations
    

In [4]:
combined_data_1_path = os.path.join(INPUT_DIR, combined_data_1)
result_pandas = pd.read_pickle(combined_data_1_path)

print(result_pandas.keys())
print(result_pandas['trial_metadata']['items'])

# This task used 3 tasks, each with 8 repitions
n_reps = 8
n_tasks = 3
movement_names = ['fist', 'fast_tripod', 'fast_finger_ext']


dict_keys(['data', 'srate', 'n_channels', 'filtered', 'pre_trigger_seconds', 'timestamp', 'stream_type', 'trial_metadata', 'session_timeline'])
[{'index': 0, 'model': 'fist.glb', 'animation': 0, 'repetitions': 8}, {'index': 1, 'model': 'fasttripodpinch.glb', 'animation': 0, 'repetitions': 8}, {'index': 2, 'model': 'fastfingerext.glb', 'animation': 0, 'repetitions': 8}]


In [5]:
session1_folder = os.path.join(INPUT_DIR, 'emg_recording_giuan_tewst_20260826_122524_divided')
session2_folder = os.path.join(INPUT_DIR, 'emg_recording_giuan_tewst2_20260826_135844_divided')



In [150]:
import glob 

def concat_movement(input_dir, n_reps):
    """
    Concatenate EMG data from multiple pickle files.

    Parameters:
    - pckl_files: list of str, the names of the pickle files to concatenate.
    - input_dir: str, the directory where the pickle files are located.

    Returns:
    - concatenated_data: np.ndarray, the concatenated EMG data.
    """
    move_files =  sorted(glob.glob(os.path.join(input_dir + '*/move*')))

    movement_dict= {}
    for movement_num, movement_type in enumerate(movement_names):
        start = movement_num * n_reps
        end = start + n_reps
        task_files = move_files[start:end]

        emg_data = [pd.DataFrame(pd.read_pickle(file_name)['data']) for file_name in task_files]
        #emg_data = pd.concat(data_load, ignore_index = True)

        duration = [pd.read_pickle(file_name)['duration_s'] for file_name in task_files]

        movement_dict[movement_type] = {
            'emg_data_list': emg_data, # list containing EMG data, length = reps
            'movement_dur': duration # list containing all durations for each rep
        }

    return movement_dict

In [149]:
session1_data = concat_movement(session1_folder, 8)
print(session1_data.keys())

print(len(session1_data['fist']['emg_data']))


dict_keys(['fist', 'fast_tripod', 'fast_finger_ext'])
8


In [177]:
def get_iso(movement_data, sampling_freq):
    """
    Isolate the middle 4 seconds of each movement, during which the movement is held isometrically

    Paramters:
    - movement_data: dictionary containing keys corresponding to movement types.
        Each key contains 'emg_data' and 'movement_dur' corresponding to emg data and durations (in s) for each repition within the movement type.
    - sampling_freq: float, sampling frequency of data collection in Hz

    Returns:
    - iso_dict: dict containing same keys as movement_data but with 'emg_data' from the middle of the movement duration +/- 2 seconds, total duration is 4 seconds of isometric movement
    """

    iso_dict = {}
    for movement in movement_data.keys():
        emg_reps = movement_data[movement]['emg_data']
        duration_reps = movement_data[movement]['movement_dur']

        emg_iso_data = []
        for emg_data, dur in zip(emg_reps, duration_reps):
            iso_start = int(((dur/2)-2) * sampling_freq)
            iso_end = int(((dur/2)+2) * sampling_freq)

            emg_iso = emg_data.iloc[:, iso_start:iso_end]
            emg_iso_data.append(emg_iso)

        combined_iso_emg = pd.concat(emg_iso_data, ignore_index = True, axis = 1)

        iso_dict[movement]= {
            'emg_iso_reps': emg_iso_data,
            'emg_iso_combined': combined_iso_emg
        }

    return(iso_dict)
    


In [178]:
session1_iso = get_iso(session1_data, 2000)
print(session1_iso['fist'].keys())
print(session1_iso['fist']['emg_iso_combined'].shape)
print(session1_iso['fist']['emg_iso_reps'][0].shape)


dict_keys(['emg_iso_reps', 'emg_iso_combined'])
(32, 64001)
(32, 8001)
